# Precision-Recall Curve — DeBERTa Soft-Label Model

**Prerequisite:** Run `error_analysis.ipynb` cells 1–35 (through the `14-LOAD` cell,
`A-0` setup, and the TF-IDF+LR baseline in `A-5`) so that `dev_probs`, `dev_labels`,
`base_probs`, `best_thresh`, and `DEVICE` are all in scope.

Or run the 14-LOAD cell in `calibration.ipynb` first, then the baseline fit below.

In [ ]:
# ── 14-LOAD: Restore all variables after a kernel restart ──────────────────
import os, random, logging
from pathlib import Path
from collections import Counter
from urllib import request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score

logging.basicConfig(level=logging.WARNING)
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED      = 777
MAX_LEN   = 256
DROPOUT   = 0.1
THRESHOLD = 0.35
CKPT_PATH = 'best_model_pcl.pt'   # adjust path as needed
DATA_DIR  = './dpm_data'          # adjust path as needed

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

class PCLClassifier(nn.Module):
    def __init__(self, model_name, hidden_size=256, dropout=DROPOUT):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        enc_hidden = self.encoder.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(enc_hidden, hidden_size), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_size, 1),
        )
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           token_type_ids=token_type_ids)
        return self.classifier(out.last_hidden_state[:, 0, :]).squeeze(-1)

class PCLDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.texts         = df['text'].tolist()
        self.binary_labels = df['label'].tolist()
        self.tokenizer     = tokenizer
        self.max_len       = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], padding='max_length', truncation=True,
                             max_length=self.max_len, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'binary_label': torch.tensor(self.binary_labels[idx], dtype=torch.long)}

# Load checkpoint
print(f'Loading checkpoint: {CKPT_PATH}')
ckpt       = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
hp         = ckpt['hyperparameters']
tokenizer  = AutoTokenizer.from_pretrained(hp['model_name'])
model      = PCLClassifier(model_name=hp['model_name'], dropout=DROPOUT).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
best_thresh = float(ckpt.get('optimal_threshold', THRESHOLD))
print(f'Loaded  |  threshold={best_thresh:.2f}')

# Reload dev DataFrame
SOFT_TARGET_MAP = {0: 0.00, 1: 0.10, 2: 0.70, 3: 0.90, 4: 1.00}
if not Path('dont_patronize_me.py').exists():
    url = 'https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py'
    with request.urlopen(url) as f, open('dont_patronize_me.py', 'w') as out:
        out.write(f.read().decode('utf-8'))
from dont_patronize_me import DontPatronizeMe

dpm = DontPatronizeMe(DATA_DIR, DATA_DIR)
dpm.load_task1()
data = dpm.train_task1_df
if 'orig_label' not in data.columns:
    raw = pd.read_csv(os.path.join(DATA_DIR, 'dontpatronizeme_pcl.tsv'), sep='\t', skiprows=4,
                      names=['row_id','par_id','keyword','country','text','orig_label'],
                      dtype={'par_id': str, 'orig_label': 'Int64'}, keep_default_na=False)
    raw['orig_label'] = raw['orig_label'].astype(int)
    data = data.merge(raw[['par_id','orig_label']], on='par_id', how='left')

teids = pd.read_csv(os.path.join(DATA_DIR, 'dev_semeval_parids-labels.csv'))
teids.par_id = teids.par_id.astype(str)
trids = pd.read_csv(os.path.join(DATA_DIR, 'train_semeval_parids-labels.csv'))
trids.par_id = trids.par_id.astype(str)

def build_df(ids_df, source_df):
    rows = []
    for parid in ids_df.par_id:
        row = source_df.loc[source_df.par_id == parid]
        if len(row) == 0: continue
        rows.append({'par_id': parid, 'keyword': row.keyword.values[0],
                     'text': row.text.values[0], 'label': int(row.label.values[0]),
                     'orig_label': int(row.orig_label.values[0])})
    return pd.DataFrame(rows)

tedf = build_df(teids, data)
trdf = build_df(trids, data)
tedf['soft_target'] = tedf['orig_label'].map(SOFT_TARGET_MAP)

# Run inference
dev_dataset = PCLDataset(tedf, tokenizer, max_len=MAX_LEN)
dev_loader  = DataLoader(dev_dataset, batch_size=32, shuffle=False, num_workers=0)
all_logits, all_labels = [], []
with torch.no_grad():
    for batch in dev_loader:
        logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        all_logits.append(logits.cpu().numpy())
        all_labels.append(batch['binary_label'].numpy())
dev_logits  = np.concatenate(all_logits)
dev_labels  = np.concatenate(all_labels)
dev_probs   = torch.sigmoid(torch.tensor(dev_logits)).numpy()
final_preds = (dev_probs >= best_thresh).astype(int)
print(f'Dev macro-F1: {f1_score(dev_labels, final_preds, average="macro", zero_division=0):.4f}')

# Fit TF-IDF + LR baseline (needed for baseline PR curve)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=50_000, sublinear_tf=True)
X_tr = _vec.fit_transform(trdf['text'])
X_te = _vec.transform(tedf['text'])
_lr  = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=SEED)
_lr.fit(X_tr, trdf['label'])
base_preds = _lr.predict(X_te)
base_probs = _lr.predict_proba(X_te)[:, 1]
print('Baseline fitted. All variables ready for PR curve.')

## Precision-Recall Curve (PCL class, Dev Set)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

os.makedirs('./figures', exist_ok=True)

precision_arr, recall_arr, pr_thresholds = precision_recall_curve(dev_labels, dev_probs)
pr_auc = auc(recall_arr, precision_arr)
print(f'DeBERTa PR-AUC: {pr_auc:.4f}')

base_p_arr, base_r_arr, _ = precision_recall_curve(dev_labels, base_probs)
base_pr_auc = auc(base_r_arr, base_p_arr)
print(f'Baseline PR-AUC: {base_pr_auc:.4f}')

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recall_arr, precision_arr, color='#0f766e', lw=2,
        label=f'DeBERTa PR curve (AUC = {pr_auc:.3f})')
ax.plot(base_r_arr, base_p_arr, color='#f59e0b', lw=1.5, linestyle='--',
        label=f'TF-IDF+LR baseline (AUC = {base_pr_auc:.3f})')
random_precision = dev_labels.mean()
ax.axhline(random_precision, color='#94a3b8', linestyle=':', lw=1,
           label=f'Random baseline (P = {random_precision:.3f})')

if len(pr_thresholds) > 0:
    ci = np.argmin(np.abs(pr_thresholds - best_thresh))
    ax.scatter(recall_arr[ci], precision_arr[ci], color='#dc2626', zorder=5, s=80,
               label=f'Chosen threshold = {best_thresh:.2f}')

ax.set_xlabel('Recall', fontsize=12); ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve (PCL class) — Dev Set', fontsize=12)
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/pr_curve.png', dpi=150)
plt.show()
print('Saved → ./figures/pr_curve.png')